# 📊 Smart Business Intelligence - Exploratory Data Analysis (EDA)

**Projet :** Smart Business Intelligence & Sales Analytics Platform  
**Auteur :** Chaima Debchi  
**Objectif :** Explorer en profondeur les performances commerciales de l'entreprise, identifier les gisements de croissance, révéler les sources de pertes financières et formuler des recommandations stratégiques chiffrées.

---

## 🎯 Questions Métier Clés
1. **Performance Globale :** Quel est le chiffre d'affaires total, le bénéfice net et le panier moyen ?
2. **Analyse Produits :** Quelles sont les catégories et sous-catégories les plus rentables ? Certains produits détruisent-ils de la valeur ?
3. **Analyse Géographique :** Quelles régions et quels États performent le mieux ? Quels territoires sont déficitaires ?
4. **Analyse Temporelle :** Quelle est l'évolution des ventes sur 4 ans ? Existe-t-il une forte saisonnalité ?
5. **Politique Tarifaire :** Quel est l'impact réel des remises (*Discounts*) sur la rentabilité ?

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path

# Style des graphiques
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
pd.set_option("display.float_format", lambda x: "%.2f" % x)

data_path = Path("../data/processed/cleaned_superstore.csv")
df = pd.read_csv(data_path)
df["Order Date"] = pd.to_datetime(df["Order Date"])
print(f"Données chargées : {df.shape[0]} lignes, {df.shape[1]} colonnes")

## 1. Vue d'Ensemble & Indicateurs Clés (KPIs)

In [ ]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()
total_orders = df["Order ID"].nunique()
total_customers = df["Customer ID"].nunique()
avg_order_value = total_sales / total_orders
overall_margin = (total_profit / total_sales) * 100

print(f"🔹 Chiffre d'Affaires Total : {total_sales:,.2f} $")
print(f"🔹 Bénéfice Net Total       : {total_profit:,.2f} $")
print(f"🔹 Marge Bénéficiaire       : {overall_margin:.2f} %")
print(f"🔹 Nombre de Commandes      : {total_orders:,}")
print(f"🔹 Nombre de Clients        : {total_customers:,}")
print(f"🔹 Panier Moyen (AOV)       : {avg_order_value:.2f} $")

## 2. Analyse des Produits et Rentabilité par Catégorie
Comparons les ventes et le bénéfice pour les 3 grandes catégories : `Furniture`, `Office Supplies`, `Technology`.

In [ ]:
cat_summary = df.groupby("Category").agg({
    "Sales": "sum",
    "Profit": "sum",
    "Quantity": "sum"
}).reset_index()
cat_summary["Margin_%"] = (cat_summary["Profit"] / cat_summary["Sales"]) * 100
cat_summary["Sales_Share_%"] = (cat_summary["Sales"] / total_sales) * 100
cat_summary["Profit_Share_%"] = (cat_summary["Profit"] / total_profit) * 100
cat_summary

> 💡 **Insight Stratégique Majeur : L'anomalie de la catégorie Mobilier (Furniture)**
>- **Furniture** représente **32,3 %** du chiffre d'affaires total (741 999 $), mais ne rapporte que **6,4 %** du profit (18 451 $), avec une marge minuscule de **2,49 %**.
>- **Technology** et **Office Supplies** sont les moteurs réels de l'entreprise avec chacune plus de **17 % de marge nette**.

### Zoom sur les 17 sous-catégories : Quels produits font perdre de l'argent ?

In [ ]:
subcat = df.groupby("Sub-Category").agg({
    "Sales": "sum",
    "Profit": "sum",
    "Discount": "mean"
}).reset_index().sort_values("Profit", ascending=False)
subcat["Margin_%"] = (subcat["Profit"] / subcat["Sales"]) * 100

# Visualisation
plt.figure(figsize=(14, 7))
colors = ["#d62728" if p < 0 else "#2ca02c" for p in subcat["Profit"]]
sns.barplot(data=subcat, x="Profit", y="Sub-Category", palette=colors)
plt.title("Bénéfice Net par Sous-Catégorie ($)", fontsize=14, fontweight="bold")
plt.xlabel("Bénéfice Net ($)")
plt.ylabel("Sous-Catégorie")
plt.axvline(0, color="black", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

> 💡 **Insight Produits :**
>- **Champions de rentabilité :** `Copiers` (55 618 $ de profit avec 37,2% de marge), `Phones` (44 516 $), `Accessories` (41 937 $) et `Paper` (34 054 $ avec 43,4% de marge !).
>- **Puits de pertes financières :**
  - `Tables` : **-17 725 $** de perte nette malgré 206 965 $ de ventes !
  - `Bookcases` : **-3 473 $** de perte nette.
  - `Supplies` : **-1 189 $** de perte nette.

## 3. Analyse Géographique : Régions et États

In [ ]:
region_summary = df.groupby("Region").agg({
    "Sales": "sum",
    "Profit": "sum"
}).reset_index()
region_summary["Margin_%"] = (region_summary["Profit"] / region_summary["Sales"]) * 100
print(region_summary)

In [ ]:
state_perf = df.groupby("State").agg({
    "Sales": "sum",
    "Profit": "sum",
    "Discount": "mean"
}).reset_index().sort_values("Profit", ascending=False)

print("=== TOP 5 ÉTATS LES PLUS RENTABLES ===")
print(state_perf.head(5))

print("\n=== TOP 5 ÉTATS LES PLUS DÉFICITAIRES ===")
print(state_perf.tail(5))

> 💡 **Insight Géographique :**
>- La région **West** est la plus performante (108 418 $ de profit, marge 14,9%).
>- La région **Central** sous-performe lourdement avec seulement **7,9 % de marge**.
>- **L'État du Texas** est le cas le plus critique : avec 170 188 $ de ventes, il génère **-25 729 $ de pertes** ! De même pour l'Ohio (-16 971 $) et la Pennsylvanie (-15 560 $).

## 4. Analyse de la Politique Tarifaire : Pourquoi ces pertes géantes ?

In [ ]:
# Découpage en tranches de remises
df["Discount_Bracket"] = pd.cut(
    df["Discount"],
    bins=[-0.01, 0.0, 0.1, 0.2, 0.3, 0.5, 0.85],
    labels=["0% (Plein tarif)", "1-10%", "11-20%", "21-30%", "31-50%", "> 50%"]
)

discount_impact = df.groupby("Discount_Bracket", observed=False).agg({
    "Sales": "sum",
    "Profit": "sum",
    "Order ID": "count"
}).reset_index()
discount_impact["Margin_%"] = (discount_impact["Profit"] / discount_impact["Sales"]) * 100
discount_impact

> 💡 **La cause racine identifiée : La spirale des remises**
>- Ventes à **0% de remise** : marge bénéficiaire moyenne saine de **+30,1%**.
>- Ventes avec **remise jusqu'à 20%** : marge positive d'environ **+14%**.
>- Dès que la remise dépasse **20%**, la marge s'effondre dans le négatif (**-33% à -86% de perte**).
>- Au Texas et dans l'Illinois, des remises de 70% à 80% sont fréquemment accordées sur les tables et fournitures, détruisant tout le profit de l'entreprise.

## 5. Analyse Temporelle & Saisonnalité

In [ ]:
yearly = df.groupby("Order Year").agg({
    "Sales": "sum",
    "Profit": "sum",
    "Order ID": "nunique"
}).reset_index()
yearly["Sales_Growth_%"] = yearly["Sales"].pct_change() * 100
yearly["Profit_Growth_%"] = yearly["Profit"].pct_change() * 100
yearly

In [ ]:
monthly_season = df.groupby("Order Month").agg({
    "Sales": "sum",
    "Profit": "sum"
}).reset_index()
month_names = ["Jan", "Fév", "Mar", "Avr", "Mai", "Juin", "Juil", "Août", "Sep", "Oct", "Nov", "Déc"]
monthly_season["Month_Name"] = month_names

plt.figure(figsize=(14, 6))
plt.plot(monthly_season["Month_Name"], monthly_season["Sales"], marker="o", linewidth=3, label="Ventes ($)", color="#1f77b4")
plt.plot(monthly_season["Month_Name"], monthly_season["Profit"], marker="s", linewidth=2, linestyle="--", label="Profit ($)", color="#2ca02c")
plt.title("Saisonnalité Mensuelle Globale des Ventes et Profits", fontsize=14, fontweight="bold")
plt.xlabel("Mois")
plt.ylabel("Montant ($)")
plt.legend()
plt.tight_layout()
plt.show()

> 💡 **Insight Temporel :**
>- Croissance globale soutenue : le CA passe de **484 k$ (2015)** à **733 k$ (2018)** (+51% en 4 ans).
>- Très forte saisonnalité au **4ème trimestre (Q4)** : les mois de **Novembre** (352 k$) et **Décembre** (325 k$) réalisent les plus gros volumes de l'année (Black Friday, achats corporatifs de fin d'exercice budgétaire).
>- Creux d'activité en **Janvier et Février** (contrecoup des fêtes de fin d'année).

## 6. Synthèse des Recommandations Stratégiques pour le Management

1. **Plafonner les remises à 20% maximum :** Arrêter immédiatement les remises à 70-80% accordées au Texas, Illinois et Ohio sur les catégories Tables et Binders.
2. **Repenser l'offre Mobilier (Tables & Bookcases) :** Renégocier les coûts d'achat auprès des fournisseurs ou augmenter le prix de vente minimum pour sortir de la zone de perte (-17 725 $ sur les tables).
3. **Capitaliser sur les champions à forte marge :** Développer le cross-selling sur les `Copiers` (marge 37%), `Accessories` (25%) et `Paper` (43%).
4. **Optimiser la logistique en Q4 :** Anticiper les stocks et les capacités d'expédition pour faire face au pic de Novembre-Décembre.